# EAHN — Kaggle Execution Notebook
Run every cell **top to bottom** in order.
- **Cell 1** auto-detects your FF++ dataset path — check the printed result before continuing.
- If auto-detect fails, set `DATA_ROOT` manually at the bottom of Cell 1.

---
## Cell 1 — Auto-detect dataset path + configuration

In [ ]:
import os, glob

# ── Auto-detect FF++ data root ────────────────────────────────────────────────
# Searches /kaggle/input/ for a directory that contains the FF++ structure.
# Looks for: original_sequences/youtube/c23/videos/  OR  manipulated_sequences/

def find_ffpp_root(search_base="/kaggle/input", max_depth=5):
    """Walk up to max_depth levels under search_base; return the dir that
    directly contains original_sequences/ and manipulated_sequences/."""
    candidates = []
    for root, dirs, _ in os.walk(search_base):
        depth = root.replace(search_base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()   # prune — don't go deeper
            continue
        has_orig  = os.path.isdir(os.path.join(root, "original_sequences"))
        has_manip = os.path.isdir(os.path.join(root, "manipulated_sequences"))
        if has_orig and has_manip:
            candidates.append(root)
    return candidates

print("Scanning /kaggle/input for FF++ directory structure...")
found = find_ffpp_root()

if found:
    AUTO_ROOT = found[0]
    print(f"  Found: {AUTO_ROOT}")
    if len(found) > 1:
        print(f"  (Also found: {found[1:]} — using first match)")
else:
    AUTO_ROOT = None
    print("  NOT FOUND. See the directory listing below and set DATA_ROOT manually.")

# ── Show full /kaggle/input tree (2 levels) so you can identify the path ─────
print("\n/kaggle/input layout (2 levels):")
for entry in sorted(glob.glob("/kaggle/input/*/*")):
    exists_mark = "[dir] " if os.path.isdir(entry) else "[file]"
    print(f"  {exists_mark} {entry}")

# ── CONFIG — override DATA_ROOT here if auto-detect was wrong ─────────────────
DATA_ROOT = AUTO_ROOT   # ← replace with a string like "/kaggle/input/foo/bar" if wrong

if DATA_ROOT is None:
    raise ValueError(
        "Could not auto-detect FF++ root. "
        "Set DATA_ROOT manually in this cell using the paths printed above."
    )

REPO_URL    = "https://github.com/umardrazbhatti/EahnCode.git"
REPO_DIR    = "/kaggle/working/EahnCode"
OUTPUT_DIR  = "/kaggle/working/outputs"
CACHE_DIR   = "/kaggle/working/.face_cache"

EPOCHS      = 10       # set to 2 for a quick smoke-test; 10+ for real results
BATCH_SIZE  = 4
NUM_WORKERS = 0        # keep 0 on Kaggle to avoid CUDA/fork issues

print("\n" + "="*50)
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"REPO_DIR   : {REPO_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"EPOCHS     : {EPOCHS}")
print("="*50)
print("If DATA_ROOT looks wrong, edit it in this cell and re-run Cell 1 only.")

---
## Cell 2 — Quick sanity check on DATA_ROOT
Prints how many `.mp4` files exist in real and fake directories.  
Both counts must be **> 0** before you continue.

In [ ]:
import glob, os

COMPRESSION   = "c23"
MANIPULATIONS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

real_dir  = os.path.join(DATA_ROOT, "original_sequences", "youtube", COMPRESSION, "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))

print(f"Real videos directory : {real_dir}")
print(f"  exists : {os.path.isdir(real_dir)}")
print(f"  count  : {len(real_vids)}")
if real_vids:
    print(f"  sample : {os.path.basename(real_vids[0])}")

print()
total_fake = 0
for method in MANIPULATIONS:
    vdir  = os.path.join(DATA_ROOT, "manipulated_sequences", method, COMPRESSION, "videos")
    count = len(glob.glob(os.path.join(vdir, "*.mp4")))
    total_fake += count
    status = "OK" if count > 0 else "MISSING"
    print(f"  [{status:7}]  {method:<18}  {count} videos")

print()
ok = len(real_vids) > 0 and total_fake > 0
if ok:
    print(f"READY — {len(real_vids)} real videos, {total_fake} fake videos found.")
else:
    print("ERROR — one or both classes are missing.")
    print("Fix DATA_ROOT in Cell 1 and re-run Cell 1 + Cell 2.")
    raise SystemExit("Stopping — fix DATA_ROOT first.")

---
## Cell 3 — Clean up previous run
Deletes old repo clone and outputs. The dataset mount under `/kaggle/input/` is **never touched**.

In [ ]:
import shutil, os

def rm(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file: {path}")
    else:
        print(f"Not found (skip): {path}")

rm(REPO_DIR)
rm(OUTPUT_DIR)
rm(CACHE_DIR)
rm("/kaggle/working/eahn_outputs.zip")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("\nWorking directory after cleanup:")
print(os.listdir("/kaggle/working"))

---
## Cell 4 — Clone repo (fresh copy from GitHub)

In [ ]:
!git clone {REPO_URL} {REPO_DIR}
!ls {REPO_DIR}

---
## Cell 5 — Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

# Confirm key packages installed correctly
import importlib
for pkg in ["torch", "torchvision", "timm", "sklearn", "matplotlib", "seaborn", "cv2"]:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg:<15} {ver}")
    except ImportError:
        print(f"  MISSING  {pkg}")

---
## Cell 6 — Full dataset verification
Runs `verify_dataset.py` which loads an actual data batch and does a model forward pass.  
Must print **ALL CHECKS PASSED** before proceeding.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable,
     f"{REPO_DIR}/scripts/verify_dataset.py",
     "--data_root", DATA_ROOT],
    capture_output=False
)

if result.returncode != 0:
    print("\n" + "="*60)
    print("VERIFICATION FAILED — stopping here.")
    print("The quick check in Cell 2 passed but the full loader failed.")
    print("Most likely cause: face-aligner dependency error OR bad frame read.")
    print("Check the traceback printed above, fix the issue, then re-run from Cell 3.")
    print("="*60)
    raise SystemExit("Fix the error above before training.")

print("\nDataset fully verified. Proceeding to training.")

---
## Cell 7 — Train + Evaluate
Runs the full pipeline: train → evaluate → graphs → heatmap videos → text explanations.

In [ ]:
os.chdir(REPO_DIR)

!python run_full_pipeline.py \
    --data_root           "{DATA_ROOT}" \
    --dataset_name        ff++ \
    --dataset_compression c23 \
    --output_dir          "{OUTPUT_DIR}" \
    --cache_dir           "{CACHE_DIR}" \
    --epochs              {EPOCHS} \
    --batch_size          {BATCH_SIZE} \
    --num_workers         {NUM_WORKERS} \
    --eval_after_train

---
## Cell 8 — Check output files produced

In [ ]:
import os

EXPECTED = [
    "metrics.csv",
    "roc_curve.png",
    "pr_curve.png",
    "confusion_matrix.png",
    "score_distribution.png",
    "best_model.pth",
]

print(f"Output directory: {OUTPUT_DIR}\n")
all_ok = True
for fname in EXPECTED:
    fpath  = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    print(f"  {'OK' if exists else 'MISSING':8}  {fname}  {'('+str(size)+' bytes)' if exists else ''}")
    if not exists:
        all_ok = False

heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if os.path.isdir(heatmap_dir):
    vids = [f for f in os.listdir(heatmap_dir) if f.endswith(".mp4")]
    print(f"\n  heatmaps/  — {len(vids)} video(s)")
    for v in sorted(vids):
        print(f"    {v}")
else:
    print("\n  heatmaps/  MISSING")

exp_dir = os.path.join(OUTPUT_DIR, "explanations")
if os.path.isdir(exp_dir):
    txts = [f for f in os.listdir(exp_dir) if f.endswith(".txt")]
    print(f"\n  explanations/  — {len(txts)} file(s)")
    for t in sorted(txts):
        print(f"    {t}")
else:
    print("\n  explanations/  MISSING")

print("\n" + ("All expected outputs present." if all_ok else "WARNING: some outputs are missing."))

---
## Cell 9 — Display detection metrics table

In [ ]:
import csv, pandas as pd
from IPython.display import display

csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
rows = []
with open(csv_path, newline="") as f:
    reader = csv.reader(f)
    next(reader, None)
    for row in reader:
        if len(row) == 2:
            try:
                rows.append({"Metric": row[0], "Value": f"{float(row[1]):.4f}"})
            except ValueError:
                rows.append({"Metric": row[0], "Value": row[1]})

print("=" * 40)
print("  EAHN Evaluation Metrics")
print("=" * 40)
display(pd.DataFrame(rows).style.set_properties(**{"text-align": "left"}))

---
## Cell 10 — Display detection graphs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

graphs = [
    ("roc_curve.png",          "ROC Curve"),
    ("pr_curve.png",           "Precision-Recall Curve"),
    ("confusion_matrix.png",   "Confusion Matrix"),
    ("score_distribution.png", "Score Distribution"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (fname, title) in zip(axes.flatten(), graphs):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        ax.imshow(mpimg.imread(fpath))
        ax.set_title(title, fontsize=13, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{fname}\nnot found", ha="center", va="center", color="red")
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Cell 11 — Display heatmap videos (first sample, all 4 methods)

In [ ]:
from IPython.display import Video, HTML, display

heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if not os.path.isdir(heatmap_dir):
    print("No heatmaps directory found.")
else:
    methods   = ["intrinsic", "gradcam", "rollout", "shap"]
    mp4_files = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".mp4"))
    video_ids = sorted(set(
        f.replace(f"_{m}.mp4", "") for f in mp4_files for m in methods
        if f.endswith(f"_{m}.mp4")
    ))

    if not video_ids:
        print("No heatmap videos found in", heatmap_dir)
    else:
        vid = video_ids[0]
        display(HTML(f"<h3>Sample: {vid}</h3>"))
        for method in methods:
            fpath = os.path.join(heatmap_dir, f"{vid}_{method}.mp4")
            if os.path.exists(fpath):
                display(HTML(f"<b>{method.upper()}</b>"))
                display(Video(fpath, embed=True, width=480))
            else:
                print(f"  [{method}] not found")
        if len(video_ids) > 1:
            print(f"\n({len(video_ids) - 1} more sample(s) in {heatmap_dir})")

---
## Cell 12 — Print plain-English explanations

In [ ]:
exp_dir = os.path.join(OUTPUT_DIR, "explanations")
if not os.path.isdir(exp_dir):
    print("No explanations directory found.")
else:
    txt_files = sorted(f for f in os.listdir(exp_dir) if f.endswith(".txt"))
    if not txt_files:
        print("No .txt files found.")
    else:
        for fname in txt_files[:3]:
            print("\n" + "=" * 60)
            print(f" {fname}")
            print("=" * 60)
            with open(os.path.join(exp_dir, fname), encoding="utf-8") as f:
                print(f.read())
        if len(txt_files) > 3:
            print(f"\n... and {len(txt_files) - 3} more in {exp_dir}")

---
## Cell 13 — Full dashboard summary

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from scripts.dashboard import show_dashboard
show_dashboard(OUTPUT_DIR)

---
## Cell 14 — Zip outputs for download
Creates `/kaggle/working/eahn_outputs.zip`. Download via the Kaggle **Output** tab.

In [ ]:
import shutil, os

zip_base = "/kaggle/working/eahn_outputs"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
full_zip = zip_base + ".zip"
size_mb  = os.path.getsize(full_zip) / 1e6
print(f"Created: {full_zip}  ({size_mb:.1f} MB)")
print("Download via Kaggle Output tab → eahn_outputs.zip")